# อ่านไฟล์ที่เก็บข้อมูลสินค้าจาก Konvy Scaper แล้ว เลือกเอา Product ID เท่านั้นไปเก็บคอมเม้น

In [ ]:
import pandas as pd

df = pd.read_excel(r"D:\konvy_data_['alese'].xlsx")
product_id_list = df['Product ID'].unique().tolist()

In [ ]:
import json
import time
import pandas as pd
import requests

# Load product IDs from CSV


# Initialize a list to store all cleaned comment data

all_cleaned_data = []
import random

user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/115.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/100.0 Safari/537.36",
]


# Define headers

headers = {
    "User-Agent": random.choice(user_agents),
    "Referer": "https://www.konvy.com/",
    "Accept": "application/json",
}


# Retry settings

max_retries = 3

retry_delay = 2  # seconds

product_delay = 5  # Delay after each product (seconds)


for product_id in product_id_list:

    page_num = 1  # Start with the first page

    total_pages_fetched = 0  # Track total pages fetched

    while True:  # Loop through pages until no comments are found
        # https://www.konvy.com/team/ajax_comment.php?action=comment_show_json_new&page=1&team_id=50090&ctype=default&score=0&onlyhaveimg=0&imgbtnclicked=0
        url = f"https://www.konvy.com/team/ajax_comment.php?action=comment_show_json_new&page={page_num}&team_id={product_id}&ctype=default&score=0&onlyhaveimg=0&imgbtnclicked=0"

        retries = 0

        while retries < max_retries:

            try:

                r = requests.get(url, headers=headers, timeout=10)
                
                raw_data = r.text
                # print(raw_data)
                # Try parsing JSON

                data = json.loads(raw_data)
                # print(data)
                # Extract comments from the data

                comments = data.get("comments", [])

                if not comments:

                    print(
                        f"No more comments found for product_id {product_id} on page {page_num}."
                    )

                    break  # Exit page loop if no comments are found

                for comment in comments:

                    user = comment.get("user", {})

                    team = comment.get("team", {})

                    all_cleaned_data.append(
                        {
                            "Product ID": product_id ,
                            "Review ID": comment.get("id"),
                            "User ID": user.get("id"),
                            "Username": user.get("username"),
                            "Skin Type": user.get("skintype"),
                            "Age": user.get("birth_age"),
                            "Review Content": comment.get("content"),
                            "Score": comment.get("score"),
                            "Review Date": comment.get("create_time"),
                            "Product Name": team.get("product"),
                            "Brand Name": team.get("brand_name"),
                            "Category": comment.get("cate_id"),
                            "Price": team.get("team_price"),
                            "Images": comment.get("images"),
                            #                         "Page Num": page_num  # Add total pages fetched
                        }
                    )

                total_pages_fetched = page_num  # Update the last fetched page number
                time.sleep(3)
                page_num += 1  # Go to the next page
                
                break  # Exit retry loop if successful

            except json.JSONDecodeError:

                print(
                    f"Failed to parse JSON for product_id {product_id} on page {page_num}. Retrying..."
                )

                retries += 1

                time.sleep(retry_delay)

            except requests.exceptions.RequestException as e:

                print(
                    f"Request error for product_id {product_id} on page {page_num}: {e}. Retrying..."
                )

                retries += 1

                time.sleep(retry_delay)

        if retries == max_retries:

            print(
                f"Failed to fetch data for product_id {product_id} on page {page_num} after {max_retries} retries."
            )

            break  # Exit page loop if max retries are reached

        if not comments:

            break  # Exit page loop if no comments are found

    # Add total page number to DataFrame for all fetched data for this product ID

    print(
        f"Completed product_id {product_id}. Total pages fetched: {total_pages_fetched}."
    )

    # print(f"Waiting for {product_delay} seconds before continuing...")

    # time.sleep(product_delay)

# df_comment

df_comment = pd.DataFrame(all_cleaned_data)

output_file = rf'D:\konvy_data\comment_data\comment_data_new_req_product.xlsx'

if os.path.exists(output_file):
    df_old = pd.read_excel(output_file)
    df_combined = pd.concat([df_old, df_comment], ignore_index=True)
    df_combined.drop_duplicates(subset=["Review ID"], inplace=True)
    df_combined.to_excel(output_file, index=False)
    print("📁 Updated existing Excel file.")
else:
    df_comment.to_excel(output_file, index=False)
    print("📁 Created new Excel file.")